In [ ]:
import logging

from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.compact_magic_state_cultivation import MagicStateCultivation
from library.qubit_array import QubitArray
from library.steane_code.compact_patch import SteaneCodePatch
from library.surface_code.patch import SurfaceCodePatch

from utils.simulation.clifft import sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 7

TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
observables = [Pauli.X, Pauli.Z]
scenarios: dict[str, Circuitry] = dict()
point = 0

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=False)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.T)

msc.append_preparation(circuitry)
msc.steane.append_noiseless_measurement(circuitry)

circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenario = rf"Prepared [$\overline{{\mathbf{{H}}}}_{{xy}}$]"
scenarios[scenario] = circuitry

circuitry.to_file(
    FILEROOT + f".point{point}.prepared"
)
display(
    Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
)

point += 1

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=False)
magic = SurfaceCodePatch(qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring))
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.T)

msc.append_preparation(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenario = rf"Superdensed [$\overline{{\mathbf{{H}}}}_{{xy}}$]"
scenarios[scenario] = circuitry

circuitry.to_file(
    FILEROOT + f".point{point}.superdense"
)
display(
    Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
)

point += 1

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=False)
magic = SurfaceCodePatch(qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring))
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.T)

msc.append_preparation(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_cultivation(circuitry)

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenario = rf"Cultivated [$\overline{{\mathbf{{H}}}}_{{xy}}$]"
scenarios[scenario] = circuitry

circuitry.to_file(
    FILEROOT + f".point{point}.cultivation"
)
display(
    Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
)

point += 1

In [ ]:
post_teleportation_extras: dict[Pauli, list[str]] = {
    Pauli.X: [
        "STN:TPT0:XB",
        "STN:TPT1:XB",
        "STN:TPT2:XB",
        "STN:DST:X1",
        "STN:DST:X5",
        "STN:DST:X6",
    ],
    Pauli.Y: [
        "JCT0:Z0",
        "JCT0:Z1",
        "JCT0:Z2",
        "STN:TPT0:XB",
        "STN:TPT1:XB",
        "STN:TPT2:XB",
        "STN:DST:X1",
        "STN:DST:X5",
        "STN:DST:X6",
    ],
    Pauli.Z: ["JCT0:Z0", "JCT0:Z1", "JCT0:Z2"],
}

In [ ]:
for observable in observables:
    qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
    circuitry = Circuitry(qubits, clifford=False)
    magic = SurfaceCodePatch(qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring))
    msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.T)

    msc.append_preparation(circuitry)
    msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
    msc.append_cultivation(circuitry)
    msc.append_teleportation(circuitry, TELEPORT_ROUNDS)

    msc.annotate_detectors(circuitry, sdc_rounds=1, tpt_rounds=TELEPORT_ROUNDS)

    circuitry.append_observable(
        0,
        f"{observable}_OBSERVABLE_TELEPORTED",
        msc.source.logical(observable),
        *post_teleportation_extras[observable]
    )

    scenario = rf"Teleported [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.teleported.{observable.name.lower()}-basis"
    )

    display(
        Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
    )

point += 1

In [ ]:
for observable in observables:
    qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
    circuitry = Circuitry(qubits, clifford=False)
    magic = SurfaceCodePatch(qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring))
    msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.T)

    msc.append_preparation(circuitry)
    msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
    msc.append_cultivation(circuitry)
    msc.append_teleportation(circuitry, TELEPORT_ROUNDS)
    msc.append_expansion(circuitry)

    msc.annotate_detectors(circuitry, sdc_rounds=1, tpt_rounds=TELEPORT_ROUNDS)

    circuitry.append_observable(
        0,
        f"{observable}_OBSERVABLE_EXPANDED",
        msc.target.logical(observable, offset=TARGET_DISTANCE-5),
        *post_teleportation_extras[observable]
    )

    scenario = rf"Expanded [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = circuitry
    circuitry.to_file(
        FILEROOT + f".point{point}.expanded.{observable.name.lower()}-basis"
    )

    display(
        Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})")
    )

point += 1

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{T}\rangle$ [Corrected $\frac{\overline{\mathbf{X}}+\overline{\mathbf{Z}}}{\sqrt{2}}$]"
sample(
    scenarios,
    title=title,
    label="Point",
    shots=1e6,
    correction=True,
    postselection=True,
    figsize=(16, 5),
    fontsize=8,
)